# ETL Pipeline: Livestock Intelligence - FASE 1 (EXTRACT)
**Laporan Praktikum Teknologi Perekayasaan Data - Kelompok 1 (3SI1)**

## Scope Extract Sesuai Arsitektur ETL
Fase ini **hanya bertugas mengekstrak (menarik) data mentah** dari berbagai sumber dan memuatnya langsung ke *Staging Area* (PostgreSQL `staging_db`) apa adanya (as-is). 
- ❌ **TIDAK ADA** normalisasi atau *unpivot*
- ❌ **TIDAK ADA** *surrogate key* (pembuatan ID baru)
- ❌ **TIDAK ADA** DDL *constraint* (Primary Key, Foreign Key)
- ❌ **TIDAK ADA** *cleaning* secara manual
- ❌ **TIDAK ADA** logika bisnis atau kalkulasi turunan


In [ ]:
# =========================================================
# IMPORT LIBRARY & KONFIGURASI DATABASE
# =========================================================
!pip insall tqdm openpyxl cryptography

import requests
import pandas as pd
import numpy as np
import random
import time
import os
from tqdm import tqdm
from IPython.display import display
from sqlalchemy import create_engine, text

import warnings
warnings.filterwarnings('ignore')
random.seed(42)
# --- KONEKSI POSTGRESQL (STAGING_DB) ---
PG_USER = 'postgres'
PG_PASS = '-RqorROOT44'
PG_HOST = 'localhost'
PG_PORT = '5432'
engine_staging = create_engine(f'postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/staging_db')

# --- KONEKSI MYSQL (ISIKHNAS_DB) ---
MYSQL_USER = 'root'
MYSQL_PASS = ''
MYSQL_HOST = 'localhost'
MYSQL_PORT = '3306'
engine_isikhnas = create_engine(f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASS}@{MYSQL_HOST}:{MYSQL_PORT}/isikhnas_db')

PIHPS_FILE = os.path.normpath(
    os.path.join(BASE_DIR, "../../DATA/PIPHPS/final_data.xlsx")
)

print("✅ Modul dan Konfigurasi Koneksi Berhasil Dimuat.")


  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 1.8 MB/s eta 0:00:02
   ------------- -------------------------- 1.3/3.8 MB 2.2 MB/s eta 0:00:02
   ------------------- -------------------- 1.8/3.8 MB 2.5 MB/s eta 0:00:01
   ---------------------- ----------------- 2.1/3.8 MB 2.2 MB/s eta 0:00:01
   ------------------------------ --------- 2.9/3.8 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 2.6 MB/s eta 0:00:00
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

   ---------------------------------------- 0/3 [pycparser]
   ------------- -------------------------- 1/3 [cffi]
   ------------- -------------------------- 1/3 [cffi]
   -------------------------- ------------- 2/3 [


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: C:\Users\ROG\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'tqdm'

## 1. EXTRACT DATA BPS (API & DUMMY)
Menarik data dari BPS SIMDASI melalui Web API. Menggunakan metode Fast Scraping yang menembak API nasional satu kali lalu mengekstrak semua provinsinya secara langsung.


In [10]:
# ============================================================
# CELL 3 · KONFIGURASI API BPS & FUNGSI INTI SCRAPING
# ============================================================

API_KEY = "d328d5f200379241367024848106698e"
YEARS   = [2020, 2021, 2022, 2023, 2024, 2025]

# -----------------------------------------------------------
# Definisi 4 komoditas yang di-scrape
# (produksi_daging_sapi TIDAK di-scrape → masuk dummy)
# -----------------------------------------------------------
COMMODITIES_TO_SCRAPE = [
    {
        "name"      : "Jumlah Penduduk",
        "table_ids" : ["WVRlTTcySlZDa3lUcFp6czNwbHl4QT09"],
        "keywords"  : ["jumlah", "penduduk"],
        "col"       : "jumlah_penduduk",
    },
    {
        "name"      : "Populasi Sapi Potong",
        "table_ids" : ["S2ViU1dwVTlpSXRwU1MvendHN05Cdz09"],
        "keywords"  : ["populasi", "sapi", "potong"],
        "col"       : "populasi_sapi",
    },
    {
        "name"      : "Populasi Ayam Pedaging",
        "table_ids" : ["ckJyVXRMT05MWTNpaW9mdnVseFk0Zz09"],
        "keywords"  : ["populasi", "ayam", "pedaging"],
        "col"       : "populasi_ayam",
    },
    {
        "name"      : "Produksi Daging Ayam Pedaging",
        "table_ids" : ["dWhmNFl6WXYyR093R2NjTGM3NG9idz09"],
        "keywords"  : ["produksi", "daging", "ayam", "pedaging"],
        "col"       : "produksi_daging_ayam",
    },
]

# -----------------------------------------------------------
# Fungsi-fungsi inti scraping
# -----------------------------------------------------------

def _get(year: int, table_id: str, kode_wilayah: str = "0000000") -> dict | None:
    """Hit satu endpoint SIMDASI BPS. Return dict JSON atau None jika gagal."""
    url = (
        f"https://webapi.bps.go.id/v1/api/interoperabilitas/datasource/simdasi/"
        f"id/25/tahun/{year}/id_tabel/{table_id}/wilayah/{kode_wilayah}/key/{API_KEY}"
    )
    try:
        r = requests.get(url, timeout=30)
        return r.json()
    except Exception:
        return None

def _find_var_id(main_data: dict, keywords: list[str]) -> str | None:
    """
    Cari ID variabel target dalam metadata kolom BPS
    berdasarkan semua kata kunci (case-insensitive).
    """
    for var_id, info in main_data.get("kolom", {}).items():
        nama = str(info.get("nama_variabel", "")).lower()
        if all(kw in nama for kw in keywords):
            return var_id
    return None

def scrape_commodity_all_provinces(commodity: dict, year: int) -> pd.DataFrame:
    """
    Ambil data satu komoditas untuk satu tahun.
    Strategi: hit API nasional → temukan tabel+variabel yang valid →
    parse semua data provinsi dari response yang sama.
    Return DataFrame dengan kolom: provinsi, tahun, <col>.
    """
    rows = []

    for tbl_id in commodity["table_ids"]:
        raw = _get(year, tbl_id)
        if not raw or raw.get("status") != "OK":
            continue

        main_data = raw.get("data", [{}, {}])[1]
        var_id = _find_var_id(main_data, commodity["keywords"])
        if not var_id:
            continue

        # Berhasil menemukan tabel & variabel yang valid
        for item in main_data.get("data", []):
            nama_prov = item.get("label", "").strip()
            kode      = item.get("kode_wilayah", "")

            # Lewati baris "Indonesia" (total nasional)
            if not kode or nama_prov.lower() in ("indonesia", ""):
                continue

            val = None
            vars_data = item.get("variables", {})
            if var_id in vars_data:
                val = vars_data[var_id].get("value_raw", None)

            rows.append({
                "provinsi": nama_prov,
                "tahun"   : year,
                commodity["col"]: val,
            })

        # Tabel valid ditemukan, tidak perlu cek tabel alternatif
        break

    return pd.DataFrame(rows)

print("✅ Fungsi scraping berhasil didefinisikan.")
print(f"   Komoditas yang akan di-scrape: {[c['name'] for c in COMMODITIES_TO_SCRAPE]}")


✅ Fungsi scraping berhasil didefinisikan.
   Komoditas yang akan di-scrape: ['Jumlah Penduduk', 'Populasi Sapi Potong', 'Populasi Ayam Pedaging', 'Produksi Daging Ayam Pedaging']


### Pre-flight Check


In [ ]:
# =========================================================
# PRE-FLIGHT CHECK
# =========================================================
print("="*60)
print(" PRE-FLIGHT CHECK API BPS ")
print("="*60)

year_to_check = 2023 
for commodity_info in COMMODITIES_TO_SCRAPE:
    commodity_name = commodity_info["name"]
    table_ids = commodity_info["table_ids"]
    keywords = commodity_info["keywords"]
    
    status_ok = False
    for tbl_id in table_ids:
        temp_data = _get(year_to_check, tbl_id, "0000000")
        if temp_data and temp_data.get("status") == "OK":
            main_data_check = temp_data.get("data", [{}, {}])[1]
            if _find_var_id(main_data_check, keywords):
                print(f"✅ {commodity_name.ljust(35)}: OK (Tabel {tbl_id})")
                status_ok = True
                break
    
    if not status_ok:
        print(f"❌ {commodity_name.ljust(35)}: GAGAL")


### Pelaksanaan Scraping BPS (Fast Mode)


In [3]:
# ============================================================
# CELL 7 · SCRAPING API BPS — FULL RUN
# ============================================================

print("🚀 Memulai scraping BPS...")
print(f"   Komoditas : {len(COMMODITIES_TO_SCRAPE)} variabel")
print(f"   Tahun     : {YEARS}")
print()

# Kumpulkan semua DataFrame per komoditas
commodity_dfs = {}

for comm in COMMODITIES_TO_SCRAPE:
    print(f"--- [{comm['name']}] ---")
    yearly_dfs = []

    for year in tqdm(YEARS, desc=f"  {comm['col']}", ncols=70):
        df_year = scrape_commodity_all_provinces(comm, year)
        if not df_year.empty:
            yearly_dfs.append(df_year)
        time.sleep(0.5)   # Jaga rate-limit

    if yearly_dfs:
        commodity_dfs[comm["col"]] = pd.concat(yearly_dfs, ignore_index=True)
        print(f"   ✅ {len(commodity_dfs[comm['col']])} baris terkumpul")
    else:
        print(f"   ❌ Tidak ada data berhasil diambil untuk {comm['name']}")

print()

# -----------------------------------------------------------
# Merge semua komoditas menjadi 1 DataFrame flat
# Key merge: provinsi + tahun
# -----------------------------------------------------------
if commodity_dfs:
    keys = list(commodity_dfs.keys())
    df_scraped = commodity_dfs[keys[0]]

    for col in keys[1:]:
        df_scraped = df_scraped.merge(
            commodity_dfs[col],
            on=["provinsi", "tahun"],
            how="outer"
        )

    # Normalisasi nama provinsi (strip spasi, title case konsisten)
    df_scraped["provinsi"] = df_scraped["provinsi"].str.strip()

    print(f"✅ Hasil scraping: {df_scraped.shape[0]} baris × {df_scraped.shape[1]} kolom")
    print(f"   Kolom: {list(df_scraped.columns)}")
    print()
    display(df_scraped.head(10))
else:
    print("⚠️  Tidak ada data scraping. Cek koneksi internet & API key.")
    df_scraped = pd.DataFrame(columns=["provinsi", "tahun",
                                        "jumlah_penduduk", "populasi_sapi",
                                        "populasi_ayam", "produksi_daging_ayam"])


🚀 Memulai scraping BPS...
   Komoditas : 4 variabel
   Tahun     : [2020, 2021, 2022, 2023, 2024, 2025]

--- [Jumlah Penduduk] ---


  jumlah_penduduk: 100%|████████████████| 6/6 [00:12<00:00,  2.01s/it]


   ✅ 216 baris terkumpul
--- [Populasi Sapi Potong] ---


  populasi_sapi: 100%|██████████████████| 6/6 [00:13<00:00,  2.25s/it]


   ✅ 224 baris terkumpul
--- [Populasi Ayam Pedaging] ---


  populasi_ayam: 100%|██████████████████| 6/6 [00:13<00:00,  2.30s/it]


   ✅ 224 baris terkumpul
--- [Produksi Daging Ayam Pedaging] ---


  produksi_daging_ayam: 100%|███████████| 6/6 [00:14<00:00,  2.34s/it]

   ✅ 220 baris terkumpul

✅ Hasil scraping: 224 baris × 6 kolom
   Kolom: ['provinsi', 'tahun', 'jumlah_penduduk', 'populasi_sapi', 'populasi_ayam', 'produksi_daging_ayam']



,provinsi,tahun,jumlah_penduduk,populasi_sapi,populasi_ayam,produksi_daging_ayam
0,Aceh,2020,"5.274,9",435.376,32.590.982,"35.935.469,00"
1,Aceh,2021,"5.333,7",455.177,34.075.587,"37.572.423,21"
2,Aceh,2022,"5.407,9",533.593,35.674.912,"42.031.398,71"
3,Aceh,2023,"5.482,5",229.295,35.758.787,"41.726.392,65"
4,Aceh,2024,"5.554,8",461.487,36.541.907,"44.521.723,67"
5,Aceh,2025,"5.626,0",466.908,37.864.269,"46.275.419,16"
6,Bali,2020,"4.317,4",550.350,71.729.771,"79.090.681,00"
7,Bali,2021,"4.362,7",558.463,68.720.589,"75.772.696,05"
8,Bali,2022,"4.415,1",380.559,78.517.219,"92.507.264,87"
9,Bali,2023,"4.404,3",344.161,76.117.189,"88.820.006,24"


### Generate Dummy BPS


In [1]:
# =========================================================
# GENERATE DUMMY BPS (TANPA LOGIKA BISNIS)
# =========================================================
provinsi_list = [
    "Aceh","Sumatera Utara","Sumatera Barat","Riau","Kepulauan Riau","Jambi",
    "Sumatera Selatan","Bangka Belitung","Bengkulu","Lampung","DKI Jakarta",
    "Jawa Barat","Jawa Tengah","DI Yogyakarta","Jawa Timur","Banten","Bali",
    "Nusa Tenggara Barat","Nusa Tenggara Timur","Kalimantan Barat",
    "Kalimantan Tengah","Kalimantan Selatan","Kalimantan Timur",
    "Kalimantan Utara","Sulawesi Utara","Sulawesi Tengah","Sulawesi Selatan",
    "Sulawesi Tenggara","Gorontalo","Sulawesi Barat","Maluku","Maluku Utara",
    "Papua","Papua Barat","Papua Selatan","Papua Tengah",
    "Papua Pegunungan","Papua Barat Daya"
]

def generate_bps_dummy():
    data = []
    for prov in provinsi_list:
        base_pop = random.randint(500000, 50000000)
        for tahun in YEARS:
            growth = round(random.uniform(-0.02, 0.04), 4)
            jumlah_penduduk = int(base_pop * (1 + growth))
            
            populasi_sapi = random.randint(10000, 500000)
            populasi_ayam = random.randint(50000, 5000000)
            
            potong_sapi = int(populasi_sapi * random.uniform(0.4, 0.7))
            potong_ayam = int(populasi_ayam * random.uniform(0.5, 0.8))
            
            produksi_sapi = round(potong_sapi * random.uniform(0.2, 0.3), 2)
            produksi_ayam = round(potong_ayam * random.uniform(0.1, 0.2), 2)
            
            konsumsi_sapi = round(random.uniform(1.5, 3.5), 2)
            konsumsi_ayam = round(random.uniform(8, 15), 2)
            
            permintaan_sapi = round(jumlah_penduduk * konsumsi_sapi / 1000, 2)
            permintaan_ayam = round(jumlah_penduduk * konsumsi_ayam / 1000, 2)
            
            harga_sapi = random.randint(90000, 130000)
            harga_ayam = random.randint(20000, 40000)
            
            row = {
                "provinsi": prov,
                "tahun": tahun,
                "jumlah_penduduk_dummy": jumlah_penduduk,
                "populasi_sapi_dummy": populasi_sapi,
                "populasi_ayam_dummy": populasi_ayam,
                "produksi_daging_sapi_dummy": produksi_sapi,
                "produksi_daging_ayam_dummy": produksi_ayam,
                "konsumsi_daging_sapi_dummy": konsumsi_sapi,
                "konsumsi_daging_ayam_dummy": konsumsi_ayam,
                "permintaan_daging_sapi_dummy": permintaan_sapi,
                "permintaan_daging_ayam_dummy": permintaan_ayam,
                "jumlah_ternak_sapi_potong_dummy": potong_sapi,
                "jumlah_ternak_ayam_potong_dummy": potong_ayam,
                "harga_baseline_sapi_dummy": harga_sapi,
                "harga_baseline_ayam_dummy": harga_ayam,
                "growth_populasi_dummy": growth
            }

            if random.random() < 0.05:
                row["produksi_daging_sapi_dummy"] = None
            if random.random() < 0.05:
                row["permintaan_daging_ayam_dummy"] = None

            data.append(row)

    return pd.DataFrame(data)

df_bps_dummy_raw = generate_bps_dummy()
print("✅ Generate dummy selesai.")


NameError: name 'random' is not defined

### Push Staging BPS


In [8]:
# =========================================================
# PUSH KE POSTGRESQL (STAGING BPS)
# =========================================================
print("📤 Mengirim data raw BPS ke staging_db...")
if not df_scraped.empty:
    df_scraped.to_sql('staging_bps_api_raw', engine_staging, if_exists='replace', index=False)
df_bps_dummy_raw.to_sql('staging_bps_dummy_raw', engine_staging, if_exists='replace', index=False)
print("✅ Tabel staging_bps_api_raw dan staging_bps_dummy_raw berhasil dimuat.")


📤 Mengirim data raw BPS ke staging_db...
✅ Tabel staging_bps_api_raw dan staging_bps_dummy_raw berhasil dimuat.


## 2. EXTRACT DATA iSIKHNAS


In [9]:
# =========================================================
# EXTRACT iSIKHNAS DARI MYSQL -> POSTGRESQL (STAGING)
# =========================================================
ISIKHNAS_TABLES = [
    "ref_hewan",
    "ref_wilayah",
    "tr_mutasi",
    "tr_laporan_sakit",
    "tr_hasil_lab",
    "tr_rph"
]

print("🔄 Mengekstrak data iSIKHNAS...")
for table in ISIKHNAS_TABLES:
    try:
        df_isikhnas = pd.read_sql(f"SELECT * FROM {table}", engine_isikhnas)
        staging_table_name = f"staging_isikhnas_{table}"
        df_isikhnas.to_sql(staging_table_name, engine_staging, if_exists='replace', index=False)
        print(f"  ✅ {table} berhasil diekstrak ({len(df_isikhnas)} baris) -> {staging_table_name}")
    except Exception as e:
        print(f"  ❌ Gagal ekstrak tabel {table}: {e}")


🔄 Mengekstrak data iSIKHNAS...
  ✅ ref_hewan berhasil diekstrak (2 baris) -> staging_isikhnas_ref_hewan
  ✅ ref_wilayah berhasil diekstrak (38 baris) -> staging_isikhnas_ref_wilayah
  ✅ tr_mutasi berhasil diekstrak (500 baris) -> staging_isikhnas_tr_mutasi
  ✅ tr_laporan_sakit berhasil diekstrak (300 baris) -> staging_isikhnas_tr_laporan_sakit
  ✅ tr_hasil_lab berhasil diekstrak (300 baris) -> staging_isikhnas_tr_hasil_lab
  ✅ tr_rph berhasil diekstrak (400 baris) -> staging_isikhnas_tr_rph


## 3. EXTRACT DATA PIHPS (HARGA HARIAN)


In [4]:
# =========================================================
# EXTRACT PIHPS (EXCEL) -> POSTGRESQL (STAGING)
# =========================================================
print("🔄 Mengekstrak data PIHPS...")
if os.path.exists(PIHPS_FILE):
    try:
        df_pihps = pd.read_excel(PIHPS_FILE)
        df_pihps.to_sql("staging_pihps_raw", engine_staging, if_exists='replace', index=False)
        print(f"  ✅ Data PIHPS berhasil diekstrak ({len(df_pihps)} baris) -> staging_pihps_raw")
    except Exception as e:
        print(f"  ❌ Gagal memuat file Excel PIHPS: {e}")
else:
    print(f"  ❌ File PIHPS tidak ditemukan di path: {PIHPS_FILE}")


🔄 Mengekstrak data PIHPS...


NameError: name 'PIHPS_FILE' is not defined

## VALIDASI STAGING AREA


In [18]:
# =========================================================
# VALIDASI TABEL DI STAGING_DB
# =========================================================
print("="*50)
print(" REKAPITULASI TABEL STAGING AREA ")
print("="*50)

try:
    with engine_staging.connect() as conn:
        tables = conn.execute(text(
            "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'"
        )).fetchall()
        
        if not tables:
            print("Tidak ada tabel ditemukan di public schema staging_db.")
        
        for t in sorted([t[0] for t in tables]):
            count = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
            print(f"- {t.ljust(30)} : {count} baris")
except Exception as e:
    print(f"Gagal melakukan validasi: {e}")


 REKAPITULASI TABEL STAGING AREA 
- staging_bps_api_raw            : 224 baris
- staging_bps_dummy_raw          : 228 baris
- staging_isikhnas_ref_hewan     : 2 baris
- staging_isikhnas_ref_wilayah   : 38 baris
- staging_isikhnas_tr_hasil_lab  : 300 baris
- staging_isikhnas_tr_laporan_sakit : 300 baris
- staging_isikhnas_tr_mutasi     : 500 baris
- staging_isikhnas_tr_rph        : 400 baris
- staging_pihps_raw              : 21966 baris
